# 1. Environment Setup & Dependencies
Installs required libraries including HuggingFace tools, pandas, spaCy, and LanguageTool for grammar validation.

In [8]:
!pip install -q transformers datasets accelerate pandas sacremoses spacy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 25.6 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 80.6 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# 2. Grammar Checker Initialization
Sets up a local LanguageTool instance. This ensures our synthetic hallucinations are semantically incorrect but grammatically valid, preventing the model from just learning to spot basic typos.

In [9]:
# Grammar-check dependency
!pip install -q language_tool_python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 3.3 MB/s eta 0:00:00


In [10]:
import language_tool_python

class GrammarChecker:
    """
    Lightweight wrapper around LanguageTool (offline) that decides whether
    a sentence is 'grammatically acceptable' for dataset inclusion.

    A sentence is REJECTED if it has more than `max_errors` grammar/style
    errors after ignoring punctuation-only and whitespace rules, which are
    too noisy for machine-generated text.

    Why this matters:
        Hallucinations should be semantically wrong (wrong entity, wrong
        number, wrong logic) but syntactically/grammatically correct,
        so the model learns to catch meaning-level errors, not typos.
    """

    # LanguageTool rule categories we want to IGNORE
    # (punctuation & typography are too strict for synthetic text)
    _IGNORE_CATEGORIES = {
        "PUNCTUATION",
        "TYPOGRAPHY",
        "WHITESPACE",
        "CASING",        # NER swaps can change casing legitimately
    }

    def __init__(self, max_errors: int = 2, language: str = "en-US"):
        """
        Args:
            max_errors:  Maximum tolerated grammar errors before rejection.
                         Default=2 gives some headroom for minor model quirks.
            language:    LanguageTool language code.
        """
        print("Loading LanguageTool grammar checker (first run downloads ~200 MB)...")
        self.tool = language_tool_python.LanguageTool(language)
        self.max_errors = max_errors
        print("GrammarChecker ready.")

    def _filter_matches(self, matches):
        """Remove noisy rule categories from LanguageTool match list."""
        return [
            m for m in matches
            if m.category.upper() not in self._IGNORE_CATEGORIES
        ]

    def is_grammatically_valid(self, text: str) -> bool:
        """
        Returns True if the sentence has <= max_errors meaningful grammar errors.
        """
        if not text or not text.strip():
            return False
        matches = self.tool.check(text)
        significant_errors = self._filter_matches(matches)
        return len(significant_errors) <= self.max_errors

    def error_count(self, text: str) -> int:
        """Returns the number of meaningful grammar errors in text."""
        matches = self.tool.check(text)
        return len(self._filter_matches(matches))


# Initialise once; reused in all generation loops below
grammar_checker = GrammarChecker(max_errors=2)

# Quick sanity test
_tests = [
    ("The Prime Minister visited Kandy on Tuesday.", True),   # correct
    ("Minister The visited on Kandy Tuesday Prime.", False),  # scrambled
    ("She go to the school yesterday.", False),               # wrong tense
]
print("\nGrammar checker sanity tests:")
for sent, expected in _tests:
    result = grammar_checker.is_grammatically_valid(sent)
    status = "✓" if result == expected else "✗"
    print(f"  {status} [{result}] '{sent}'")


Loading LanguageTool grammar checker (first run downloads ~200 MB)...


GrammarChecker ready.

Grammar checker sanity tests:
  ✓ [True] 'The Prime Minister visited Kandy on Tuesday.'
  ✗ [True] 'Minister The visited on Kandy Tuesday Prime.'
  ✗ [True] 'She go to the school yesterday.'


# 3. Load & Clean Source Dataset
Loads the NLPC-UOM Sinhala-English dataset. Cleans the data by removing rows where the English target contains Sinhala characters or lacks English letters, then extracts a sample for generation.

In [11]:
import pandas as pd
import re
from datasets import load_dataset

# 1. Load the NLPC-UOM cleaned dataset
print("Loading NLPC-UOM/nllb-top25k-ensi-cleaned...")
dataset = load_dataset("NLPC-UOM/nllb-top25k-ensi-cleaned", split="train")

# 2. Convert to Pandas
df = pd.DataFrame(dataset)

# 3. Rename columns to match your preferred format
# Mapping: SI_sentence -> sinhala, EN_sentence -> english
df.rename(columns={
    'SI_sentence': 'sinhala', 
    'EN_sentence': 'english'
}, inplace=True)

# --- CLEANING STEP START ---
print(f"Original Row Count: {len(df)}")

def is_valid_english(text):
    """
    Returns True only if:
    1. The text DOES NOT contain Sinhala characters (Unicode range \u0D80-\u0DFF).
    2. The text DOES contain at least one English letter (a-z, A-Z).
    """
    s = str(text)
    # Check for Sinhala characters
    if re.search(r'[\u0D80-\u0DFF]', s):
        return False
    # Check for at least one English letter (avoids numbers/punctuation only rows)
    if not re.search(r'[a-zA-Z]', s):
        return False
    return True

# Apply filtering
df = df[df['english'].apply(is_valid_english)].copy()
print(f"Cleaned Row Count: {len(df)} (Removed mixed/invalid rows)")
# --- CLEANING STEP END ---

# 4. Take a sample of 5000 rows from the CLEANED data
# We use reset_index so the indices start fresh from 0
df_sample = df.sample(n=7500, random_state=42).reset_index(drop=True)

print(f"Final Sample Size: {len(df_sample)}")
print("Columns found:", df_sample.columns.tolist())
display(df_sample.head())

Loading NLPC-UOM/nllb-top25k-ensi-cleaned...


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Original Row Count: 25000
Cleaned Row Count: 24996 (Removed mixed/invalid rows)
Final Sample Size: 7500
Columns found: ['english', 'sinhala', 'cosine_similarity']


,english,sinhala,cosine_similarity
0,"Nowadays, you do not call them miracles.","වර්තමානයේ, ඔබ ඒවා ප්‍රාතිහාර්යයන් ලෙස නොකියයි.",0.868030
1,And their son is an exact replica of his father.,ඒ වගේම ඔවුන්ගේ පුතා ඔහුගේ පියාගේම අනුරුවකි.,0.781679
2,Trust us and you will get more.,"අපව විශ්වාස කරන්න, එවිට ඔබට තවත් වාසි ලැබෙනු ඇත.",0.851481
3,"Channel-tag, together with the world!","නාලිකා පාඨය, ලෝකය සමඟ එක්ව!",0.802776
4,It is the beast that we promised you.,අපි ඔබට පොරොන්දු වූ සත්ත්වයා මොහු යි.,0.792051


# 4. Load Base Translation Model (NLLB)
Loads the NLLB 1.3B model and tokenizer into GPU memory. This is used later for the high-temperature stochastic generation method.

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Setup Device (GPU is highly recommended for NLLB 1.3B)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Load the NLLB Model and Tokenizer
model_checkpoint = "facebook/nllb-200-distilled-1.3B"

print(f"Loading model: {model_checkpoint} (this might take a minute)...")

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint,
                                             torch_dtype=torch.float16).to(device)

print("Model and Tokenizer loaded successfully.")



Using device: cuda
Loading model: facebook/nllb-200-distilled-1.3B (this might take a minute)...


config.json:   0%|          | 0.00/808 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.48G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1016 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model and Tokenizer loaded successfully.


# 6. Execute Dataset Generation Pipeline
The main execution loop. It generates 4 negative samples per ground-truth row by:
1. Planning the mutations (80% temp / 20% atomic).
2. Executing GPU-batched generation for the high-temperature samples.
3. Applying secondary CPU-based mutations (NER/Semantic/etc.) to further corrupt the text.

In [16]:
import spacy
import random
import re
import nltk
import torch
from collections import defaultdict
from typing import List, Dict, Tuple


try:
    from nltk.corpus import wordnet
except ImportError:
    import nltk
    nltk.download('wordnet')
    from nltk.corpus import wordnet


class HallucinationGenerator:
    def __init__(self, reference_sentences: List[str], model=None, tokenizer=None, device='cuda'):
        self.sentences = [str(s) for s in reference_sentences if str(s) != 'nan']
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        
        # --- Load Spacy ---
        try:
            self.nlp = spacy.load("en_core_web_sm")
        except OSError:
            print("Downloading Spacy model...")
            from spacy.cli import download
            download("en_core_web_sm")
            self.nlp = spacy.load("en_core_web_sm")

        # --- Build Entity Pool ---
        print("Building Entity Pool for NER Swaps...")
        self.entity_pool = self._build_entity_pool()
        print(f"Ready. Entity Pool contains {sum(len(v) for v in self.entity_pool.values())} entities.")

    def _build_entity_pool(self) -> Dict[str, List[str]]:
        pool = defaultdict(list)
        sample_size = min(len(self.sentences), 2000) 
        for doc in self.nlp.pipe(random.sample(self.sentences, sample_size)):
            for ent in doc.ents:
                clean_text = ent.text.strip()
                if len(clean_text) > 1:
                    pool[ent.label_].append(clean_text)
        return pool

    # --- ATOMIC METHODS (Optimized to reuse 'doc') ---
    def hallucinate_ner_swap(self, text: str, doc=None) -> str:
        if doc is None: doc = self.nlp(text)
        if not doc.ents: return text 
        target_ent = random.choice(doc.ents)
        label = target_ent.label_
        if label not in self.entity_pool or not self.entity_pool[label]: return text
        candidates = self.entity_pool[label]
        replacement = random.choice(candidates)
        if replacement == target_ent.text and len(candidates) > 1:
            replacement = random.choice(candidates)
        return text.replace(target_ent.text, replacement, 1)

    def hallucinate_semantic_swap(self, text: str, doc=None) -> str:
        # Semantic swap doesn't rely strictly on Spacy deps, so we just use the text
        words = text.split()
        if not words: return text
        for _ in range(3):
            target_idx = random.randint(0, len(words)-1)
            word = words[target_idx]
            antonyms = []
            for syn in wordnet.synsets(word):
                for lemma in syn.lemmas():
                    if lemma.antonyms():
                        antonyms.append(lemma.antonyms()[0].name())
            if antonyms:
                replacement = random.choice(antonyms)
                if word[0].isupper(): replacement = replacement.capitalize()
                words[target_idx] = replacement.replace('_', ' ')
                return " ".join(words)
        return text

    def hallucinate_dependency_swap(self, text: str, doc=None) -> str:
        if doc is None: doc = self.nlp(text)
        subj, obj = None, None
        for token in doc:
            if "subj" in token.dep_: subj = token
            if "obj" in token.dep_: obj = token
        if subj and obj and subj.text.lower() != obj.text.lower():
            placeholder = "___TEMP_SWAP___"
            temp_text = text.replace(subj.text, placeholder, 1)
            temp_text = temp_text.replace(obj.text, subj.text, 1)
            temp_text = temp_text.replace(placeholder, obj.text, 1)
            return temp_text
        return text

    def hallucinate_number_swap(self, text: str, doc=None) -> str:
        numbers = re.findall(r'\b\d+\b', text)
        if not numbers: return text
        target = random.choice(numbers)
        val = int(target)
        strategy = random.choice(['inc', 'dec', 'repeat'])
        if strategy == 'inc':
            new_val = val + random.randint(1, max(10, int(val*0.2)))
        elif strategy == 'dec':
            new_val = max(0, val - random.randint(1, max(10, int(val*0.2))))
        else:
            new_val = int(str(val) + str(random.randint(0,9)))
        return text.replace(target, str(new_val), 1)

    def _detect_applicable_strategies(self, text: str, doc=None) -> Tuple[List[callable], object]:
        """Returns applicable strategies AND the parsed doc to save future computation."""
        if doc is None: doc = self.nlp(text)
        strategies = []
        if doc.ents: strategies.append(self.hallucinate_ner_swap)
        if re.search(r'\b\d+\b', text): strategies.append(self.hallucinate_number_swap)
        if len(text.split()) > 4: strategies.append(self.hallucinate_semantic_swap)
        subj = any("subj" in t.dep_ for t in doc)
        obj = any("obj" in t.dep_ for t in doc)
        if subj and obj: strategies.append(self.hallucinate_dependency_swap)
        return strategies, doc

# --- NEW: BATCHED GPU GENERATION ---
    def hallucinate_high_temp_batch(self, texts: List[str], temperature: float = 1.5, batch_size: int = 32) -> List[str]:
        """Processes a list of sentences through the LLM all at once."""
        from tqdm.auto import tqdm # Import tqdm here just to be safe
        
        if not self.model or not self.tokenizer or not texts: 
            return texts
            
        results = []
        forced_bos_token_id = self.tokenizer.convert_tokens_to_ids("eng_Latn")
        
        # ---> WRAP THE RANGE IN TQDM <---
        for i in tqdm(range(0, len(texts), batch_size), desc="GPU Batch Generation"):
            batch = texts[i:i+batch_size]
            inputs = self.tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(self.device)
            
            gen_kwargs = {
                "do_sample": True, "temperature": temperature, "top_k": 50, "top_p": 0.95, 
                "max_length": 128, "forced_bos_token_id": forced_bos_token_id 
            }
            
            with torch.no_grad(): # Saves memory during generation
                outputs = self.model.generate(**inputs, **gen_kwargs)
                
            results.extend(self.tokenizer.batch_decode(outputs, skip_special_tokens=True))
            
        return results

In [18]:
import pandas as pd
import random
from tqdm import tqdm

# Initialize
hallucinator = HallucinationGenerator(
    reference_sentences=df_sample['english'].tolist(),
    model=model,
    tokenizer=tokenizer,
    device=device
)

data_rows = []
temp_generation_queue = [] # Holds texts that need LLM generation

print("Phase 1: Planning dataset and applying base CPU mutations...")

# We will do 4 hallucinations per row
num_hallucinations_per_row = 4

for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    sinhala_src = row['sinhala']
    english_gt = row['english'] 
    
    # --- A. Correct Sample ---
    data_rows.append({
        "sinhala": sinhala_src,
        "hypothesis": english_gt,       
        "reference": english_gt,        
        "label": 0,
        "method": "ground_truth",
        "needs_temp": False 
    })
    
    # --- B. Plan Negative Samples ---
    for _ in range(num_hallucinations_per_row):
        current_text = english_gt
        applied_labels = []
        needs_temp = False
        
        # STEP 1: BASE DECISION (80% Temp / 20% Atomic)
        if random.random() < 0.8:
            needs_temp = True
            applied_labels.append("temp")
            # We will queue this for the GPU later. For now, keep the text as english_gt.
        else:
            strategies, doc = hallucinator._detect_applicable_strategies(current_text)
            if strategies:
                func = random.choice(strategies)
                current_text = func(current_text, doc=doc)
                name = func.__name__.replace("hallucinate_", "").replace("_swap", "")
                applied_labels.append(name)
            else:
                needs_temp = True
                applied_labels.append("temp")

        # Record the plan
        row_plan = {
            "sinhala": sinhala_src,
            "hypothesis": current_text, # Will be updated later if needs_temp is True
            "reference": english_gt,
            "label": 1,
            "method": " + ".join(applied_labels),
            "needs_temp": needs_temp,
            "base_text_for_temp": english_gt # What we feed to the LLM
        }
        data_rows.append(row_plan)

# Create a working DataFrame from the plan
working_df = pd.DataFrame(data_rows)

# --- Phase 2: EXECUTE GPU BATCHING ---
print("\nPhase 2: Executing Batched GPU Generation...")
temp_mask = working_df['needs_temp'] == True
texts_to_generate = working_df.loc[temp_mask, 'base_text_for_temp'].tolist()

if texts_to_generate:
    # This is where the magic happens. 32 sentences at a time instead of 1!
    generated_texts = hallucinator.hallucinate_high_temp_batch(texts_to_generate, temperature=1.5, batch_size=32)
    working_df.loc[temp_mask, 'hypothesis'] = generated_texts

# --- Phase 3: APPLY SECONDARY CPU MUTATIONS ---
print("\nPhase 3: Applying Secondary Mutations...")

# Convert back to list of dicts for faster row iteration
final_rows = working_df.to_dict('records')

for row in tqdm(final_rows):
    if row['label'] == 1: # Only mutate hallucinations
        # 80% chance for a second mutation
        if random.random() < 0.8:
            current_text = row['hypothesis']
            strategies, doc = hallucinator._detect_applicable_strategies(current_text)
            
            # Avoid repeating the exact same type
            existing_labels = row['method'].split(" + ")
            valid_strategies = [f for f in strategies if f.__name__.replace("hallucinate_", "").replace("_swap", "") not in existing_labels]
            
            if valid_strategies:
                func = random.choice(valid_strategies)
                new_text = func(current_text, doc=doc)
                
                if new_text != current_text:
                    row['hypothesis'] = new_text
                    name = func.__name__.replace("hallucinate_", "").replace("_swap", "")
                    row['method'] += f" + {name}"

        # Final check: Did it actually change?
        if row['hypothesis'] == row['reference']:
            row['label'] = 0
            row['method'] = "ground_truth (fallback)"

# --- Final Output ---
synthetic_df = pd.DataFrame(final_rows)
synthetic_df = synthetic_df.drop(columns=['needs_temp', 'base_text_for_temp']) # Clean up helper columns
synthetic_df = synthetic_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nDataset Generation Complete!")
print("Dataset Shape:", synthetic_df.shape)
display(synthetic_df.head())

Building Entity Pool for NER Swaps...
Ready. Entity Pool contains 1059 entities.
Phase 1: Planning dataset and applying base CPU mutations...


100%|██████████| 7500/7500 [00:39<00:00, 190.03it/s]


Phase 2: Executing Batched GPU Generation...


GPU Batch Generation:   0%|          | 0/755 [00:00<?, ?it/s]


Phase 3: Applying Secondary Mutations...


100%|██████████| 37500/37500 [02:23<00:00, 261.57it/s]



Dataset Generation Complete!
Dataset Shape: (37500, 5)


,sinhala,hypothesis,reference,label,method
0,"එසේම, අශ්වයන් බටහිරට වඩා වෙනස් ආකාරයකින් පුහුණ...","Also, the horses are trained differently than ...","Also, the horses are trained differently than ...",0,ground_truth
1,"අපේ ණයවලට සමාව දෙන්න,",and forgive us our debts...,and forgive us our debts...,0,ground_truth
2,ධර්මිෂ්ඨ අයට එළඹෙන දිනයන්හි බිය වීමට කිසිවක් නැත.,The justifying days aren't like in the future ...,The righteous have nothing to fear in the days...,1,temp + ner
3,ඔඩ්නොක්ලාස්නිකි හි ඔබේ පිටුවට ඉක්මනින් ඇතුළු ව...,How about entering your page off Odnoklasse-ni...,"To quickly enter your page on Odnoklassniki, a...",1,temp + semantic
4,"කණගාටුවට කරුණක් නම්, වසර තුනකට වඩා වැඩි කාලයක්...",Sadly for over three years tchildreny have so ...,"Sadly, for more than three years, they almost ...",1,temp + dependency


In [19]:
synthetic_df.to_csv("synthetic_hallucinations_probabilistic.csv")

# 7. Balance Dataset & Export
Downsamples the majority class to create a perfectly balanced dataset (50% faithful, 50% hallucinated) and provides links to download both the full and balanced CSV files.

In [20]:
# ==========================================================
# DOWNLOAD DATASETS
# Saves both the full dataset and a class-balanced version,
# then triggers a browser download for each (works on Kaggle
# and Google Colab; on local Jupyter the files are in the CWD).
# ==========================================================
import pandas as pd
from IPython.display import FileLink, display as ipy_display

# ── 1. Full dataset (already saved above as CSV) ───────────────────────────
full_path = "synthetic_hallucinations_probabilistic.csv"
print(f"Full dataset: {len(synthetic_df)} rows")
print(f"  Class balance:\n{synthetic_df['label'].value_counts().to_string()}\n")

# ── 2. Balanced dataset (downsample majority class) ────────────────────────
df_faithful      = synthetic_df[synthetic_df['label'] == 0]
df_hallucination = synthetic_df[synthetic_df['label'] == 1]

# Match minority class size
min_count = min(len(df_faithful), len(df_hallucination))
df_balanced = pd.concat([
    df_faithful.sample(n=min_count, random_state=42),
    df_hallucination.sample(n=min_count, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

balanced_path = "synthetic_hallucinations_probabilistic_balanced.csv"
df_balanced.to_csv(balanced_path, index=False)
print(f"Balanced dataset: {len(df_balanced)} rows")
print(f"  Class balance:\n{df_balanced['label'].value_counts().to_string()}\n")

# ── 3. Download links ──────────────────────────────────────────────────────
# Works on Kaggle / Colab. On local Jupyter, files are already in the CWD.
try:
    # Kaggle: copy to /kaggle/working so the output panel picks them up
    import shutil, os
    kaggle_out = "/kaggle/working"
    if os.path.exists(kaggle_out):
        shutil.copy(full_path,     os.path.join(kaggle_out, full_path))
        shutil.copy(balanced_path, os.path.join(kaggle_out, balanced_path))
        print("Files copied to /kaggle/working — visible in the Kaggle Output panel.")
    else:
        raise FileNotFoundError("Not on Kaggle")
except Exception:
    # Colab / local fallback
    try:
        from google.colab import files
        print("Triggering Colab downloads...")
        files.download(full_path)
        files.download(balanced_path)
    except ImportError:
        # Local Jupyter: just show clickable links
        print("Local Jupyter — click to download:")
        ipy_display(FileLink(full_path))
        ipy_display(FileLink(balanced_path))

print("\nDone! Both CSVs are ready.")


Full dataset: 37500 rows
  Class balance:
label
1    29376
0     8124

Balanced dataset: 16248 rows
  Class balance:
label
0    8124
1    8124

Triggering Colab downloads...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Done! Both CSVs are ready.


In [21]:
!pip install -q pandarallel

  Preparing metadata (setup.py) ... done


In [22]:
import pandas as pd
from tqdm.auto import tqdm
import language_tool_python

# Enable tqdm for pandas apply
tqdm.pandas(desc="Grammar Check Progress")

print("Starting local LanguageTool server (just once!)...")
# Initialize the Java server exactly ONCE outside the function
tool = language_tool_python.LanguageTool("en-US")
IGNORE_CATEGORIES = {"PUNCTUATION", "TYPOGRAPHY", "WHITESPACE", "CASING"}

def check_grammar_stable(text):
    """Checks grammar using the globally initialized tool."""
    if not isinstance(text, str) or not text.strip():
        return False
        
    try:
        matches = tool.check(text)
        significant_errors = [m for m in matches if m.category.upper() not in IGNORE_CATEGORIES]
        return len(significant_errors) <= 2
    except Exception:
        # If anything weird happens, default to False to be safe
        return False

# We only need to check rows that are currently labeled as hallucinations
print("Running stable grammar check on generated hypotheses...")

synthetic_df = pd.read_csv("/kaggle/input/datasets/navamobeysekara/final-synthetic/synthetic_hallucinations_exhaustive(2).csv")
mask = synthetic_df['label'] == 1

# Use .progress_apply() to see exactly where it's at!
valid_grammar_mask = synthetic_df.loc[mask, 'hypothesis'].progress_apply(check_grammar_stable)

# Update the dataset based on the results
invalid_indices = valid_grammar_mask[~valid_grammar_mask].index

# Revert those failed rows back to faithful ground truth
synthetic_df.loc[invalid_indices, 'label'] = 0
synthetic_df.loc[invalid_indices, 'hypothesis'] = synthetic_df.loc[invalid_indices, 'reference']
synthetic_df.loc[invalid_indices, 'method'] = "ground_truth (grammar_rejected)"

# Safely close the server now that we are completely done with it
tool.close()

print(f"Grammar check complete! Rejected {len(invalid_indices)} grammatically broken rows.")

Starting local LanguageTool server (just once!)...
Running stable grammar check on generated hypotheses...


Grammar Check Progress:   0%|          | 0/24996 [00:00<?, ?it/s]

Grammar check complete! Rejected 79 grammatically broken rows.


In [23]:
# --- Save the updated dataset ---
output_file_path = "/kaggle/working/synthetic_hallucinations_grammar_checked.csv"

# index=False prevents pandas from writing the row numbers as an extra column
synthetic_df.to_csv(output_file_path, index=False)

print(f"Successfully saved the updated dataset to: {output_file_path}")

Successfully saved the updated dataset to: /kaggle/working/synthetic_hallucinations_grammar_checked.csv
